# Build a Budget with alarm, simulate spend with a tagged workload, and use Cost Explorer to identify the cost driver

A startup's AWS bill went from $4,200 in March to $16,800 in April with no team-recalled architectural changes. The CFO has paused engineering hiring until the team can explain the spike and put guardrails in place before the next billing cycle closes. You are the engineer who has 65 minutes to ship the guardrails: a Budget at the right threshold, an Anomaly Detection monitor on the same scope, a clean tag-attribution path so the team can group spend by workload, and a working Cost Explorer query you hand to finance as the new monthly close report. The lab does not produce the April spike answer because Cost Explorer's 24-hour data lag means the bill from this session shows up tomorrow; what the lab DOES produce is the shape of the guardrails that catch the next spike before it costs $12,000.

You will build five things in order:

1. SNS topic with an email subscription that backs the budget alarm and the anomaly subscription.
2. AWS Budget at a $1 cost ceiling, scoped by the `labrun:lab-slug` tag, with an SNS subscriber.
3. Cost Anomaly Detection monitor on the same tag with a Cost Anomaly Detection subscription delivering to the SNS topic.
4. A tagged workload (t4g.nano EC2 + S3 bucket + S3 object) so Cost Explorer has tagged resources to group by.
5. A Cost Explorer `GetCostAndUsage` call with `GroupBy=[{Type: TAG, Key: labrun:lab-slug}]` that returns the call shape Cost Explorer publishes; cost values may be zero because of the 24-hour data lag.

**Time.** Plan for about 65 minutes hands-on. EC2 launch and instance status checks add 2-3 minutes; the rest is filling small forms and reading the response shapes.

**Cost.** This lab costs about two cents. The interesting line item is the Cost Explorer API call: it is one of the few AWS APIs that charges per request, at one cent each. The thing this lab actually produces is the cost discipline that catches the next $12,000 mistake before it happens.

**Interaction mode.** This lab runs in `config` mode (see `content/labs/INTERACTION-MODES.md`). Each task cell renders an ipywidgets form. Validators always query AWS state directly, never widget state.

This lab maps to AWS SAA-C03 Domain 4 (Design Cost-Optimized Architectures, 20% of exam weight). It is the capstone for the SAA cert track.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks and ipywidgets.

!pip install --quiet labrun-checks==0.6.0 ipywidgets

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12.

import atexit
import datetime as dt
import getpass
import json
import re
import time

import boto3
from botocore.exceptions import ClientError
import ipywidgets as widgets
from IPython.display import display, clear_output

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "cost-explorer-and-budgets-end-to-end"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

SNS_TOPIC_NAME = f"labrun-{LAB_ID}-topic"
BUDGET_NAME = f"labrun-{LAB_ID}-budget"
ANOMALY_MONITOR_NAME = f"labrun-{LAB_ID}-anomaly-monitor"
ANOMALY_SUBSCRIPTION_NAME = f"labrun-{LAB_ID}-anomaly-subscription"
EC2_NAME = f"labrun-{LAB_ID}-ec2"
EC2_ROLE_NAME = f"labrun-{LAB_ID}-ec2-role"
INSTANCE_PROFILE_NAME = f"labrun-{LAB_ID}-instance-profile"
SG_NAME = f"labrun-{LAB_ID}-sg"
BUCKET_NAME = None  # resolved after STS get_caller_identity
OBJECT_KEY = "workload/sample.log"

LAB_STATE = {
    "sns_topic_arn": None,
    "sns_subscription_arn": None,
    "anomaly_monitor_arn": None,
    "anomaly_subscription_arn": None,
    "ec2_instance_id": None,
    "sg_id": None,
    "cost_explorer_response": None,
}

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials, resolve bucket name and
# account ID. Both Budgets and the SNS topic policy need the account ID.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"SAA labs run in {REGION} (N. Virginia).")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit, orphan scan.
# Budgets, Cost Anomaly Detection monitor, and Cost Anomaly Detection
# subscription have no adapter in labrun-checks 0.6.0; the cleanup cell
# deletes them manually before run_cleanup runs the rest of the manifest.

CLEANUP_MANIFEST: list[CleanupResource] = []


def _rebuild_manifest():
    CLEANUP_MANIFEST.clear()
    if BUCKET_NAME:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="s3_object",
                id=OBJECT_KEY,
                parent=BUCKET_NAME,
                region=REGION,
                cli_delete_command=(
                    f"aws s3api delete-object --bucket {BUCKET_NAME} --key {OBJECT_KEY}"
                ),
            )
        )
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="s3_bucket",
                id=BUCKET_NAME,
                region=REGION,
                cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
            )
        )
    if LAB_STATE["ec2_instance_id"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="ec2_instance",
                id=LAB_STATE["ec2_instance_id"],
                region=REGION,
                cli_delete_command=(
                    f"aws ec2 terminate-instances --instance-ids {LAB_STATE['ec2_instance_id']}"
                ),
            )
        )
    if LAB_STATE["sg_id"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="ec2_security_group",
                id=LAB_STATE["sg_id"],
                region=REGION,
                cli_delete_command=f"aws ec2 delete-security-group --group-id {LAB_STATE['sg_id']}",
            )
        )
    CLEANUP_MANIFEST.append(
        CleanupResource(
            type="iam_instance_profile",
            id=INSTANCE_PROFILE_NAME,
            parent=EC2_ROLE_NAME,
            region=REGION,
            cli_delete_command=(
                f"aws iam remove-role-from-instance-profile "
                f"--instance-profile-name {INSTANCE_PROFILE_NAME} "
                f"--role-name {EC2_ROLE_NAME} && "
                f"aws iam delete-instance-profile "
                f"--instance-profile-name {INSTANCE_PROFILE_NAME}"
            ),
        )
    )
    CLEANUP_MANIFEST.append(
        CleanupResource(
            type="iam_role",
            id=EC2_ROLE_NAME,
            region=REGION,
            cli_delete_command=f"aws iam delete-role --role-name {EC2_ROLE_NAME}",
        )
    )
    if LAB_STATE["sns_topic_arn"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="sns_topic",
                id=LAB_STATE["sns_topic_arn"],
                region=REGION,
                cli_delete_command=(
                    f"aws sns delete-topic --topic-arn {LAB_STATE['sns_topic_arn']}"
                ),
            )
        )


_rebuild_manifest()


def _delete_manually_managed_resources():
    """Delete the Budget, Cost Anomaly subscription, and Cost Anomaly monitor.
    labrun-checks 0.6.0 has no adapter for any of them; the cleanup cell
    handles them inline before run_cleanup runs."""
    ce = boto3.client(
        "ce",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name="us-east-1",
    )
    budgets = boto3.client(
        "budgets",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name="us-east-1",
    )
    # Cost Anomaly subscription must be deleted BEFORE its monitor.
    if LAB_STATE.get("anomaly_subscription_arn"):
        try:
            ce.delete_anomaly_subscription(
                SubscriptionArn=LAB_STATE["anomaly_subscription_arn"],
            )
        except ClientError:
            pass
    if LAB_STATE.get("anomaly_monitor_arn"):
        try:
            ce.delete_anomaly_monitor(
                MonitorArn=LAB_STATE["anomaly_monitor_arn"],
            )
        except ClientError:
            pass
    try:
        budgets.delete_budget(AccountId=ACCOUNT_ID, BudgetName=BUDGET_NAME)
    except ClientError:
        pass


def _atexit_cleanup():
    try:
        _delete_manually_managed_resources()
        _rebuild_manifest()
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to provision the cost-control surfaces.")

## Task 1: Create the SNS topic with the lab tag and an email subscription

SNS backs both the budget alarm action and the Cost Anomaly Detection subscription. Without it, Budgets and Anomaly Detection have nowhere to send messages. The topic policy must grant `sns:Publish` to `costalerts.amazonaws.com` so Cost Anomaly Detection can deliver. The form handles this for you.

Type your email address into the form. AWS sends a confirmation email; clicking it from your inbox moves the subscription from `PendingConfirmation` to `Confirmed`. The checkpoint accepts the pending state because the click is out of band.

In [ ]:
# NBVAL_SKIP
# Task 1 widget: email input + Apply -> create_topic + tag + topic policy + subscribe.

email_field = widgets.Text(
    value="", description="Email:",
    placeholder="you@example.com",
    layout=widgets.Layout(width="500px"),
)
apply_task1 = widgets.Button(description="Apply", button_style="success")
out_task1 = widgets.Output()

def _on_apply_task1(_):
    with out_task1:
        clear_output()
        email = email_field.value.strip()
        if not re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", email):
            print(f"Email {email!r} does not look valid. Try again.")
            return
        sns = boto3.client(
            "sns",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            topic = sns.create_topic(
                Name=SNS_TOPIC_NAME,
                Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
            )
            LAB_STATE["sns_topic_arn"] = topic["TopicArn"]
            print(f"Created SNS topic: {topic['TopicArn']}")
        except ClientError as e:
            print(f"create_topic failed: {e}")
            return

        topic_policy = {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Sid": "AllowCostAlertsPublish",
                    "Effect": "Allow",
                    "Principal": {"Service": "costalerts.amazonaws.com"},
                    "Action": "sns:Publish",
                    "Resource": LAB_STATE["sns_topic_arn"],
                },
                {
                    "Sid": "AllowBudgetsPublish",
                    "Effect": "Allow",
                    "Principal": {"Service": "budgets.amazonaws.com"},
                    "Action": "sns:Publish",
                    "Resource": LAB_STATE["sns_topic_arn"],
                },
            ],
        }
        try:
            sns.set_topic_attributes(
                TopicArn=LAB_STATE["sns_topic_arn"],
                AttributeName="Policy",
                AttributeValue=json.dumps(topic_policy),
            )
            print("Topic policy grants sns:Publish to costalerts and budgets service principals.")
        except ClientError as e:
            print(f"set_topic_attributes failed: {e}")
            return

        try:
            sub = sns.subscribe(
                TopicArn=LAB_STATE["sns_topic_arn"],
                Protocol="email",
                Endpoint=email,
                ReturnSubscriptionArn=True,
            )
            LAB_STATE["sns_subscription_arn"] = sub.get("SubscriptionArn")
            print(f"Subscribed {email}. SubscriptionArn: {LAB_STATE['sns_subscription_arn']}")
            print("Check your email for the AWS confirmation link. Until you click it,")
            print("the subscription stays in PendingConfirmation. The checkpoint tolerates that.")
        except ClientError as e:
            print(f"subscribe failed: {e}")
        _rebuild_manifest()

apply_task1.on_click(_on_apply_task1)

display(widgets.VBox([
    widgets.HTML("<b>Task 1: SNS topic + tag + email subscription</b>"),
    email_field,
    apply_task1,
    out_task1,
]))

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: SNS topic exists with the lab tag and an email subscription.

def checkpoint_1(session):
    try:
        sns = boto3.client(
            "sns",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        topic_arn = f"arn:aws:sns:{REGION}:{ACCOUNT_ID}:{SNS_TOPIC_NAME}"
        # Confirm the topic exists by listing all topics and matching by ARN.
        topics = []
        paginator = sns.get_paginator("list_topics")
        for page in paginator.paginate():
            for t in page.get("Topics", []):
                topics.append(t["TopicArn"])
        if topic_arn not in topics:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"SNS topic {topic_arn} not found. Click Apply on Task 1 to "
                    f"create the topic."
                ),
            )
        LAB_STATE["sns_topic_arn"] = topic_arn
        # Verify the lab tag is on the topic.
        try:
            tags = sns.list_tags_for_resource(ResourceArn=topic_arn)
            tagset = {t["Key"]: t["Value"] for t in tags.get("Tags", [])}
            if tagset.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"SNS topic exists but tag {LAB_TAG_KEY}={LAB_TAG_VALUE} is missing. "
                        f"Tag the topic at create_topic time."
                    ),
                )
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"Could not read SNS topic tags: {e}",
            )
        # Verify at least one email subscription with a plausible address.
        subs = sns.list_subscriptions_by_topic(TopicArn=topic_arn).get("Subscriptions", [])
        email_subs = [
            s for s in subs
            if s.get("Protocol") == "email"
            and re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", s.get("Endpoint", ""))
        ]
        if not email_subs:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "No email subscription found on the topic. Re-run Task 1 with "
                    "a valid email address. The subscription's ARN can be "
                    "PendingConfirmation; that is OK."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The Apply button wires `create_topic`, `set_topic_attributes` (the topic policy), and `subscribe`. If the checkpoint fails, scroll up to the widget output: the printed error usually identifies the missing call.

</details>

<details><summary>Hint 2 (direction)</summary>

Make sure the email field has a real-looking address. `create_topic` tags the topic via the `Tags` argument; without that, the lab tag is missing and the checkpoint fails on `list_tags_for_resource`. The subscription's `SubscriptionArn` is `PendingConfirmation` until you click the AWS email link, which is expected.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
topic = sns.create_topic(
    Name=SNS_TOPIC_NAME,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
sns.subscribe(
    TopicArn=topic["TopicArn"],
    Protocol="email",
    Endpoint="you@example.com",
    ReturnSubscriptionArn=True,
)
```

</details>

**Wallet check.** Effectively zero. SNS topic creation, tagging, and email subscription are free; publishing is metered but the lab does not publish.

## Task 2: Create the AWS Budget with a $1 threshold scoped by the lab tag

AWS Budgets uses a slightly oddly-shaped tag filter: it expects `user:<TagKey>$<TagValue>` for user-defined tags, not the boto3 tag shape you see elsewhere. The widget builds the filter string for you.

The form lets you tune the threshold (default $1) and the notification threshold percent (default 80%). At those defaults, the budget fires when actual spend on tagged resources reaches 80 cents in the calendar month.

In [ ]:
# NBVAL_SKIP
# Task 2 widget: budget threshold + notification percent + Apply -> create_budget +
# create_notification + create_subscriber wired to the SNS topic.

budget_amount_field = widgets.FloatText(
    value=1.0, description="Budget $:", layout=widgets.Layout(width="250px"),
)
notification_pct_field = widgets.IntText(
    value=80, description="Notify at %:", layout=widgets.Layout(width="250px"),
)
apply_task2 = widgets.Button(description="Apply", button_style="success")
out_task2 = widgets.Output()

def _on_apply_task2(_):
    with out_task2:
        clear_output()
        if not LAB_STATE.get("sns_topic_arn"):
            print("Task 1 has not run yet. Click Apply on Task 1 first to create the SNS topic.")
            return
        budgets_client = boto3.client(
            "budgets",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name="us-east-1",
        )
        budget_body = {
            "BudgetName": BUDGET_NAME,
            "BudgetLimit": {
                "Amount": f"{float(budget_amount_field.value):.2f}",
                "Unit": "USD",
            },
            "CostFilters": {
                "TagKeyValue": [f"user:{LAB_TAG_KEY}${LAB_TAG_VALUE}"],
            },
            "TimeUnit": "MONTHLY",
            "BudgetType": "COST",
        }
        notification = {
            "NotificationType": "ACTUAL",
            "ComparisonOperator": "GREATER_THAN",
            "Threshold": float(notification_pct_field.value),
            "ThresholdType": "PERCENTAGE",
            "NotificationState": "OK",
        }
        subscriber = {
            "SubscriptionType": "SNS",
            "Address": LAB_STATE["sns_topic_arn"],
        }
        try:
            budgets_client.create_budget(
                AccountId=ACCOUNT_ID,
                Budget=budget_body,
                NotificationsWithSubscribers=[
                    {"Notification": notification, "Subscribers": [subscriber]}
                ],
            )
            print(f"Created budget {BUDGET_NAME} at "
                  f"${budget_amount_field.value:.2f} with {notification_pct_field.value}% notification.")
            print(f"Tag filter: user:{LAB_TAG_KEY}${LAB_TAG_VALUE}")
            print(f"SNS subscriber: {LAB_STATE['sns_topic_arn']}")
        except ClientError as e:
            code_str = e.response.get("Error", {}).get("Code", "")
            if code_str == "DuplicateRecordException":
                print(f"Budget {BUDGET_NAME} already exists. Delete it from cleanup first if you want to recreate.")
            else:
                print(f"create_budget failed: {e}")

apply_task2.on_click(_on_apply_task2)

display(widgets.VBox([
    widgets.HTML("<b>Task 2: AWS Budget with tag filter + SNS alarm action</b>"),
    budget_amount_field,
    notification_pct_field,
    apply_task2,
    out_task2,
]))

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Budget exists with the right tag filter and an SNS subscriber.

def checkpoint_2(session):
    try:
        budgets_client = boto3.client(
            "budgets",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name="us-east-1",
        )
        try:
            resp = budgets_client.describe_budget(
                AccountId=ACCOUNT_ID, BudgetName=BUDGET_NAME,
            )
        except ClientError as e:
            code_str = e.response.get("Error", {}).get("Code", "")
            if code_str in ("NotFoundException",):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Budget {BUDGET_NAME} not found. Click Apply on Task 2."
                    ),
                )
            raise
        b = resp.get("Budget", {})
        limit = b.get("BudgetLimit", {})
        if limit.get("Unit") != "USD":
            return CheckpointResult(
                status="fail",
                error_reason=f"Budget unit is {limit.get('Unit')!r}, expected 'USD'.",
            )
        if b.get("TimeUnit") != "MONTHLY":
            return CheckpointResult(
                status="fail",
                error_reason=f"TimeUnit is {b.get('TimeUnit')!r}, expected 'MONTHLY'.",
            )
        if b.get("BudgetType") != "COST":
            return CheckpointResult(
                status="fail",
                error_reason=f"BudgetType is {b.get('BudgetType')!r}, expected 'COST'.",
            )
        cost_filters = b.get("CostFilters", {})
        tag_filters = cost_filters.get("TagKeyValue", [])
        expected_tag = f"user:{LAB_TAG_KEY}${LAB_TAG_VALUE}"
        if expected_tag not in tag_filters:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"CostFilters TagKeyValue must include {expected_tag!r}. "
                    f"Found: {tag_filters!r}."
                ),
            )

        notifications = budgets_client.describe_notifications_for_budget(
            AccountId=ACCOUNT_ID, BudgetName=BUDGET_NAME,
        ).get("Notifications", [])
        if not notifications:
            return CheckpointResult(
                status="fail",
                error_reason="Budget has no notifications. Re-click Apply on Task 2.",
            )
        found_sns = False
        for n in notifications:
            subs = budgets_client.describe_subscribers_for_notification(
                AccountId=ACCOUNT_ID, BudgetName=BUDGET_NAME, Notification=n,
            ).get("Subscribers", [])
            for s in subs:
                if (
                    s.get("SubscriptionType") == "SNS"
                    and s.get("Address") == LAB_STATE.get("sns_topic_arn")
                ):
                    found_sns = True
        if not found_sns:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No notification subscriber points at the lab SNS topic "
                    f"{LAB_STATE.get('sns_topic_arn')!r}."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Three Budgets calls run inside the Apply handler: `create_budget`, plus the notification + subscriber wired in the same call via `NotificationsWithSubscribers`. If the checkpoint complains about the SNS subscriber, scroll up and confirm Task 1 created the SNS topic first.

</details>

<details><summary>Hint 2 (direction)</summary>

AWS Budgets uses a special string format for user-defined tag filters: `user:<TagKey>$<TagValue>`. So `labrun:lab-slug=cost-explorer-and-budgets-end-to-end` becomes `user:labrun:lab-slug$cost-explorer-and-budgets-end-to-end`. Don't forget the `user:` prefix or the `$` separator.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
budgets.create_budget(
    AccountId=ACCOUNT_ID,
    Budget={
        "BudgetName": BUDGET_NAME,
        "BudgetLimit": {"Amount": "1.00", "Unit": "USD"},
        "CostFilters": {
            "TagKeyValue": [f"user:{LAB_TAG_KEY}${LAB_TAG_VALUE}"],
        },
        "TimeUnit": "MONTHLY",
        "BudgetType": "COST",
    },
    NotificationsWithSubscribers=[{
        "Notification": {
            "NotificationType": "ACTUAL",
            "ComparisonOperator": "GREATER_THAN",
            "Threshold": 80.0,
            "ThresholdType": "PERCENTAGE",
        },
        "Subscribers": [{
            "SubscriptionType": "SNS",
            "Address": LAB_STATE["sns_topic_arn"],
        }],
    }],
)
```

</details>

**Wallet check.** First two budgets per AWS account per month are free; this is your first. Still effectively zero.

## Task 3: Create the Cost Anomaly Detection monitor and subscription

Cost Anomaly Detection is the ML-driven cost-control surface that complements AWS Budgets. Budgets fires when actual spend crosses a fixed threshold; Anomaly Detection fires when spend deviates from the learned baseline by a configurable percent. Both have their place.

The form below has no parameters because the lab pins the monitor scope (tag-filtered) and the subscription threshold ($10) to known-good values. Click Apply to create the monitor and the subscription against the SNS topic from Task 1.

In [ ]:
# NBVAL_SKIP
# Task 3 widget: Apply -> create_anomaly_monitor + create_anomaly_subscription.

apply_task3 = widgets.Button(description="Apply", button_style="success")
out_task3 = widgets.Output()

def _on_apply_task3(_):
    with out_task3:
        clear_output()
        if not LAB_STATE.get("sns_topic_arn"):
            print("Task 1 has not run yet. Click Apply on Task 1 first.")
            return
        ce = boto3.client(
            "ce",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name="us-east-1",
        )
        monitor_spec = {
            "Tags": {
                "Key": LAB_TAG_KEY,
                "Values": [LAB_TAG_VALUE],
                "MatchOptions": ["EQUALS"],
            }
        }
        try:
            mon = ce.create_anomaly_monitor(
                AnomalyMonitor={
                    "MonitorName": ANOMALY_MONITOR_NAME,
                    "MonitorType": "CUSTOM",
                    "MonitorSpecification": json.dumps(monitor_spec),
                },
            )
            LAB_STATE["anomaly_monitor_arn"] = mon["MonitorArn"]
            print(f"Created anomaly monitor: {mon['MonitorArn']}")
        except ClientError as e:
            code_str = e.response.get("Error", {}).get("Code", "")
            if code_str == "ValidationException" and "duplicate" in str(e).lower():
                print("Monitor with this name already exists. Looking it up.")
                monitors = ce.get_anomaly_monitors().get("AnomalyMonitors", [])
                for m in monitors:
                    if m.get("MonitorName") == ANOMALY_MONITOR_NAME:
                        LAB_STATE["anomaly_monitor_arn"] = m["MonitorArn"]
                        break
            else:
                print(f"create_anomaly_monitor failed: {e}")
                return

        try:
            sub = ce.create_anomaly_subscription(
                AnomalySubscription={
                    "SubscriptionName": ANOMALY_SUBSCRIPTION_NAME,
                    "MonitorArnList": [LAB_STATE["anomaly_monitor_arn"]],
                    "Subscribers": [
                        {"Type": "SNS", "Address": LAB_STATE["sns_topic_arn"]},
                    ],
                    "Frequency": "IMMEDIATE",
                    "ThresholdExpression": {
                        "Dimensions": {
                            "Key": "ANOMALY_TOTAL_IMPACT_ABSOLUTE",
                            "Values": ["10"],
                            "MatchOptions": ["GREATER_THAN_OR_EQUAL"],
                        },
                    },
                },
            )
            LAB_STATE["anomaly_subscription_arn"] = sub["SubscriptionArn"]
            print(f"Created anomaly subscription: {sub['SubscriptionArn']}")
            print("Anomaly Detection is free; both surfaces are now wired to SNS.")
        except ClientError as e:
            print(f"create_anomaly_subscription failed: {e}")

apply_task3.on_click(_on_apply_task3)

display(widgets.VBox([
    widgets.HTML("<b>Task 3: Cost Anomaly monitor + subscription</b>"),
    apply_task3,
    out_task3,
]))

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Cost Anomaly monitor + subscription exist on the lab tag.

def checkpoint_3(session):
    try:
        ce = boto3.client(
            "ce",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name="us-east-1",
        )
        monitors = ce.get_anomaly_monitors().get("AnomalyMonitors", [])
        lab_monitor = next(
            (m for m in monitors if m.get("MonitorName") == ANOMALY_MONITOR_NAME), None
        )
        if lab_monitor is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Anomaly monitor {ANOMALY_MONITOR_NAME!r} not found. "
                    f"Click Apply on Task 3."
                ),
            )
        if lab_monitor.get("MonitorType") != "CUSTOM":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"MonitorType is {lab_monitor.get('MonitorType')!r}, expected 'CUSTOM' "
                    f"so the MonitorSpecification can target a specific tag."
                ),
            )
        spec_raw = lab_monitor.get("MonitorSpecification")
        if not spec_raw:
            return CheckpointResult(
                status="fail",
                error_reason="Anomaly monitor MonitorSpecification is missing.",
            )
        try:
            spec = json.loads(spec_raw) if isinstance(spec_raw, str) else spec_raw
        except json.JSONDecodeError:
            spec = {}
        tags_block = spec.get("Tags", {}) if isinstance(spec, dict) else {}
        if (
            tags_block.get("Key") != LAB_TAG_KEY
            or LAB_TAG_VALUE not in tags_block.get("Values", [])
        ):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Anomaly monitor specification does not target the lab tag. "
                    f"Expected Key={LAB_TAG_KEY!r} and Values containing {LAB_TAG_VALUE!r}."
                ),
            )
        LAB_STATE["anomaly_monitor_arn"] = lab_monitor["MonitorArn"]

        subs = ce.get_anomaly_subscriptions().get("AnomalySubscriptions", [])
        lab_sub = None
        for s in subs:
            if (
                lab_monitor["MonitorArn"] in s.get("MonitorArnList", [])
                and any(
                    sub.get("Type") == "SNS"
                    and sub.get("Address") == LAB_STATE.get("sns_topic_arn")
                    for sub in s.get("Subscribers", [])
                )
            ):
                lab_sub = s
                break
        if lab_sub is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "No anomaly subscription references the lab monitor + lab SNS topic. "
                    "Verify Task 1 created the SNS topic and Task 3 wired the subscription."
                ),
            )
        LAB_STATE["anomaly_subscription_arn"] = lab_sub["SubscriptionArn"]
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Two `ce` calls under the Apply button: `create_anomaly_monitor` followed by `create_anomaly_subscription`. The monitor's specification has to target the lab tag exactly; the subscription has to reference the monitor's ARN.

</details>

<details><summary>Hint 2 (direction)</summary>

The `MonitorSpecification` is a JSON-encoded string in the `create_anomaly_monitor` call. The Tags block requires `Key`, `Values` (a list), and `MatchOptions` (use `EQUALS`). For `create_anomaly_subscription`, the `Subscribers` list expects `Type=SNS` and `Address=<topic ARN>`, and `MonitorArnList` is the list with one entry for this lab.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
spec = json.dumps({
    "Tags": {
        "Key": LAB_TAG_KEY,
        "Values": [LAB_TAG_VALUE],
        "MatchOptions": ["EQUALS"],
    }
})
mon = ce.create_anomaly_monitor(AnomalyMonitor={
    "MonitorName": ANOMALY_MONITOR_NAME,
    "MonitorType": "CUSTOM",
    "MonitorSpecification": spec,
})
ce.create_anomaly_subscription(AnomalySubscription={
    "SubscriptionName": ANOMALY_SUBSCRIPTION_NAME,
    "MonitorArnList": [mon["MonitorArn"]],
    "Subscribers": [{"Type": "SNS", "Address": LAB_STATE["sns_topic_arn"]}],
    "Frequency": "IMMEDIATE",
    "ThresholdExpression": {
        "Dimensions": {
            "Key": "ANOMALY_TOTAL_IMPACT_ABSOLUTE",
            "Values": ["10"],
            "MatchOptions": ["GREATER_THAN_OR_EQUAL"],
        },
    },
})
```

</details>

**Wallet check.** Cost Anomaly Detection monitors and subscriptions are free. Still effectively zero.

## Task 4: Launch the tagged workload (t4g.nano + S3 bucket + S3 object)

This is the workload Cost Explorer will eventually attribute spend to. Every taggable resource carries `labrun:lab-slug=cost-explorer-and-budgets-end-to-end` so the Budgets tag filter, the anomaly monitor's tag-scoped spec, and the Cost Explorer GroupBy=TAG query all converge on the same set of ARNs.

The form below has one parameter: the EC2 instance type (defaults to t4g.nano, the cheapest ARM option at $0.0042/hour). The Apply button does all four operations: IAM role + instance profile, security group, EC2 launch, and S3 bucket + object with the lab tag.

In [ ]:
# NBVAL_SKIP
# Task 4 widget: instance type + Apply -> IAM + SG + EC2 + S3 bucket + object.

instance_type_field = widgets.Dropdown(
    options=["t4g.nano", "t4g.micro", "t3.nano", "t3.micro"],
    value="t4g.nano",
    description="Instance:",
    layout=widgets.Layout(width="300px"),
)
apply_task4 = widgets.Button(description="Apply", button_style="success")
out_task4 = widgets.Output()

def _on_apply_task4(_):
    with out_task4:
        clear_output()
        iam = boto3.client(
            "iam",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        ec2 = boto3.client(
            "ec2",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        ssm = boto3.client(
            "ssm",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        s3 = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        # IAM role + instance profile.
        trust = {
            "Version": "2012-10-17",
            "Statement": [{
                "Effect": "Allow",
                "Principal": {"Service": "ec2.amazonaws.com"},
                "Action": "sts:AssumeRole",
            }],
        }
        try:
            iam.create_role(
                RoleName=EC2_ROLE_NAME,
                AssumeRolePolicyDocument=json.dumps(trust),
                Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
            )
        except ClientError as e:
            if e.response.get("Error", {}).get("Code") != "EntityAlreadyExists":
                print(f"create_role failed: {e}")
                return
        iam.attach_role_policy(
            RoleName=EC2_ROLE_NAME,
            PolicyArn="arn:aws:iam::aws:policy/AmazonSSMManagedInstanceCore",
        )
        try:
            iam.create_instance_profile(
                InstanceProfileName=INSTANCE_PROFILE_NAME,
                Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
            )
        except ClientError as e:
            if e.response.get("Error", {}).get("Code") != "EntityAlreadyExists":
                print(f"create_instance_profile failed: {e}")
                return
        try:
            iam.add_role_to_instance_profile(
                InstanceProfileName=INSTANCE_PROFILE_NAME, RoleName=EC2_ROLE_NAME
            )
        except ClientError as e:
            if e.response.get("Error", {}).get("Code") != "LimitExceeded":
                print(f"add_role_to_instance_profile failed: {e}")
                return

        # Default VPC + first subnet.
        default_vpc = ec2.describe_vpcs(
            Filters=[{"Name": "is-default", "Values": ["true"]}],
        )["Vpcs"][0]
        subnet = ec2.describe_subnets(
            Filters=[{"Name": "vpc-id", "Values": [default_vpc["VpcId"]]}],
        )["Subnets"][0]
        subnet_id = subnet["SubnetId"]

        # Security group.
        try:
            sg = ec2.create_security_group(
                GroupName=SG_NAME,
                Description="labrun cost-explorer lab SG",
                VpcId=default_vpc["VpcId"],
                TagSpecifications=[{
                    "ResourceType": "security-group",
                    "Tags": [
                        {"Key": "Name", "Value": SG_NAME},
                        {"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE},
                    ],
                }],
            )
            LAB_STATE["sg_id"] = sg["GroupId"]
        except ClientError as e:
            if e.response.get("Error", {}).get("Code") == "InvalidGroup.Duplicate":
                LAB_STATE["sg_id"] = ec2.describe_security_groups(
                    Filters=[{"Name": "group-name", "Values": [SG_NAME]}],
                )["SecurityGroups"][0]["GroupId"]
            else:
                print(f"create_security_group failed: {e}")
                return

        chosen = instance_type_field.value
        ami_param = (
            "/aws/service/ami-amazon-linux-latest/al2023-ami-kernel-default-arm64"
            if chosen.startswith("t4g")
            else "/aws/service/ami-amazon-linux-latest/al2023-ami-kernel-default-x86_64"
        )
        ami_id = ssm.get_parameter(Name=ami_param)["Parameter"]["Value"]
        print(f"AMI: {ami_id} (for {chosen})")

        print("Waiting 15s for IAM instance profile to propagate.")
        time.sleep(15)

        try:
            resp = ec2.run_instances(
                ImageId=ami_id,
                InstanceType=chosen,
                MinCount=1,
                MaxCount=1,
                SubnetId=subnet_id,
                SecurityGroupIds=[LAB_STATE["sg_id"]],
                IamInstanceProfile={"Name": INSTANCE_PROFILE_NAME},
                TagSpecifications=[{
                    "ResourceType": "instance",
                    "Tags": [
                        {"Key": "Name", "Value": EC2_NAME},
                        {"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE},
                    ],
                }],
            )
            LAB_STATE["ec2_instance_id"] = resp["Instances"][0]["InstanceId"]
            print(f"Launched EC2 instance: {LAB_STATE['ec2_instance_id']} ({chosen})")
        except ClientError as e:
            print(f"run_instances failed: {e}")
            return

        # S3 bucket + object.
        try:
            if REGION == "us-east-1":
                s3.create_bucket(Bucket=BUCKET_NAME)
            else:
                s3.create_bucket(
                    Bucket=BUCKET_NAME,
                    CreateBucketConfiguration={"LocationConstraint": REGION},
                )
        except ClientError as e:
            if e.response.get("Error", {}).get("Code") not in ("BucketAlreadyOwnedByYou",):
                print(f"create_bucket failed: {e}")
                return
        s3.put_bucket_tagging(
            Bucket=BUCKET_NAME,
            Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
        )
        body = b"sample log line for the cost attribution lab\n" * 10
        s3.put_object(Bucket=BUCKET_NAME, Key=OBJECT_KEY, Body=body)
        s3.put_object_tagging(
            Bucket=BUCKET_NAME,
            Key=OBJECT_KEY,
            Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
        )
        print(f"S3 bucket tagged: {BUCKET_NAME}")
        print(f"S3 object tagged: s3://{BUCKET_NAME}/{OBJECT_KEY}")
        _rebuild_manifest()

apply_task4.on_click(_on_apply_task4)

display(widgets.VBox([
    widgets.HTML("<b>Task 4: tagged workload (EC2 + S3)</b>"),
    instance_type_field,
    apply_task4,
    out_task4,
]))

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: tagged EC2 + tagged S3 bucket + tagged S3 object are all live
# and surface in the resource-tagging API.

def checkpoint_4(session):
    try:
        ec2c = boto3.client(
            "ec2",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        s3c = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        tag_client = boto3.client(
            "resourcegroupstaggingapi",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        insts = ec2c.describe_instances(
            Filters=[
                {"Name": f"tag:{LAB_TAG_KEY}", "Values": [LAB_TAG_VALUE]},
                {"Name": "instance-state-name", "Values": ["pending", "running"]},
            ],
        ).get("Reservations", [])
        flat = [i for r in insts for i in r.get("Instances", [])]
        if len(flat) != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Expected exactly 1 tagged running EC2 instance, found {len(flat)}."
                ),
            )
        if flat[0]["State"]["Name"] != "running":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"EC2 state is {flat[0]['State']['Name']!r}, expected running. "
                    f"Give the instance another 30-60 seconds and re-run the checkpoint."
                ),
            )
        try:
            b_tags = s3c.get_bucket_tagging(Bucket=BUCKET_NAME)
            tagset = {t["Key"]: t["Value"] for t in b_tags.get("TagSet", [])}
            if tagset.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"S3 bucket {BUCKET_NAME!r} missing lab tag.",
                )
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"S3 bucket tagging unreadable: {e}",
            )
        try:
            s3c.head_object(Bucket=BUCKET_NAME, Key=OBJECT_KEY)
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Object s3://{BUCKET_NAME}/{OBJECT_KEY} missing. "
                    f"put_object did not run. {e}"
                ),
            )
        try:
            o_tags = s3c.get_object_tagging(Bucket=BUCKET_NAME, Key=OBJECT_KEY)
            o_tagset = {t["Key"]: t["Value"] for t in o_tags.get("TagSet", [])}
            if o_tagset.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Object {OBJECT_KEY!r} missing the lab tag. Call "
                        f"put_object_tagging separately; bucket tags do not propagate."
                    ),
                )
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"S3 object tagging unreadable: {e}",
            )
        # Resource-tagging API: must surface at least the EC2 instance and SNS topic.
        # (Bucket and Object are also tagged but may take a moment to index.)
        arns = []
        paginator = tag_client.get_paginator("get_resources")
        for page in paginator.paginate(
            TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
        ):
            for r in page.get("ResourceTagMappingList", []):
                arns.append(r.get("ResourceARN", ""))
        has_ec2 = any(":instance/" in a for a in arns)
        if not has_ec2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Resource-tagging API did not surface the tagged EC2 instance "
                    "yet. Tag indexing can take 30-60 seconds; re-run the checkpoint."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The Apply button kicks off six AWS calls behind the scenes (IAM role, instance profile, security group, run_instances, create_bucket, put_object) plus three tag calls. If the checkpoint says the EC2 is not running yet, give it 30-60 seconds and re-run.

</details>

<details><summary>Hint 2 (direction)</summary>

Object-level tags are a separate API call from bucket-level tags. `put_bucket_tagging` does NOT propagate down to objects; you must call `put_object_tagging` after each `put_object`. If Checkpoint 4 fails on object tagging, scroll up to the widget output and confirm the put_object_tagging line ran.

</details>

<details><summary>Hint 3 (spoiler)</summary>

The Apply handler already runs all the right calls. The fail mode the checkpoint usually catches is timing: EC2 takes ~30 seconds to reach `running`. Wait, then re-run the checkpoint cell.

```python
s3.put_object(Bucket=BUCKET_NAME, Key=OBJECT_KEY, Body=b"sample log")
s3.put_object_tagging(
    Bucket=BUCKET_NAME,
    Key=OBJECT_KEY,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
```

</details>

**Wallet check.** t4g.nano just started billing at $0.0042/hour ($0.0001/minute). For a 30-minute remaining window that lands at about 0.2 cents. The S3 bucket + tiny object adds fractions of a millicent. Running total still well under a cent.

## Task 5: Call Cost Explorer GetCostAndUsage grouped by tag

The final piece: the query the team will run every month to close the books. The form below sets the time window (defaults to yesterday only since Cost Explorer's 24-hour lag means today's data is not yet published) and the GroupBy tag key (defaults to `labrun:lab-slug`).

Click Apply to issue `GetCostAndUsage` with `Granularity=DAILY`, `Metrics=['UnblendedCost']`, and `GroupBy=[{Type: TAG, Key: labrun:lab-slug}]`. The response prints to the cell output. Note the cost values for the lab's own resources may be zero in this response because of the 24-hour lag; that is expected. The checkpoint validates the call shape, not the cost number.

In [ ]:
# NBVAL_SKIP
# Task 5 widget: time window + tag key + Apply -> ce.get_cost_and_usage.

today = dt.date.today()
yesterday = today - dt.timedelta(days=1)

start_field = widgets.DatePicker(
    description="Start:", value=yesterday,
    layout=widgets.Layout(width="300px"),
)
end_field = widgets.DatePicker(
    description="End:", value=today,
    layout=widgets.Layout(width="300px"),
)
tag_key_field = widgets.Text(
    value=LAB_TAG_KEY, description="Tag key:",
    layout=widgets.Layout(width="400px"),
)
apply_task5 = widgets.Button(description="Apply", button_style="success")
out_task5 = widgets.Output()

def _on_apply_task5(_):
    with out_task5:
        clear_output()
        ce = boto3.client(
            "ce",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name="us-east-1",
        )
        start = start_field.value.isoformat() if start_field.value else yesterday.isoformat()
        end = end_field.value.isoformat() if end_field.value else today.isoformat()
        tag_key = tag_key_field.value.strip() or LAB_TAG_KEY
        try:
            resp = ce.get_cost_and_usage(
                TimePeriod={"Start": start, "End": end},
                Granularity="DAILY",
                Metrics=["UnblendedCost"],
                GroupBy=[{"Type": "TAG", "Key": tag_key}],
            )
            LAB_STATE["cost_explorer_response"] = resp
            results = resp.get("ResultsByTime", [])
            print(f"Cost Explorer query returned {len(results)} ResultsByTime entries.")
            print(f"Time window: {start} to {end} (DAILY granularity)")
            print(f"GroupBy: TAG={tag_key}")
            print()
            for r in results:
                print(f"  Date {r.get('TimePeriod', {}).get('Start')}:")
                groups = r.get("Groups", [])
                if not groups:
                    print("    (no groups; Cost Explorer 24-hour lag has not surfaced data yet)")
                for g in groups:
                    keys = g.get("Keys", [])
                    amount = g.get("Metrics", {}).get("UnblendedCost", {}).get("Amount")
                    print(f"    {keys} -> ${amount}")
        except ClientError as e:
            print(f"get_cost_and_usage failed: {e}")

apply_task5.on_click(_on_apply_task5)

display(widgets.VBox([
    widgets.HTML("<b>Task 5: Cost Explorer GetCostAndUsage</b>"),
    start_field,
    end_field,
    tag_key_field,
    apply_task5,
    out_task5,
]))

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: Cost Explorer GetCostAndUsage call shape is valid; response
# parses. Per the cert YAML risks section and Gap 4 in the research doc,
# this checkpoint does NOT assert a non-zero cost or a specific tag value
# in the response because of the 24-hour Cost Explorer data lag.

def checkpoint_5(session):
    try:
        resp = LAB_STATE.get("cost_explorer_response")
        if resp is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Cost Explorer response not captured. Click Apply on Task 5 "
                    "to issue the GetCostAndUsage call before this checkpoint."
                ),
            )
        results = resp.get("ResultsByTime")
        if results is None or not isinstance(results, list):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Cost Explorer response missing ResultsByTime list. "
                    "Re-run Task 5 with a non-empty TimePeriod."
                ),
            )
        if len(results) == 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "ResultsByTime is empty. Ensure the TimePeriod spans at least "
                    "one full day (Start before End)."
                ),
            )
        first = results[0]
        if "TimePeriod" not in first or "Groups" not in first:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "ResultsByTime[0] missing TimePeriod or Groups fields. "
                    "The call shape did not include GroupBy=[{Type: TAG, ...}]."
                ),
            )
        # GroupDefinitions reflects what we asked for; verify TAG grouping was used.
        group_defs = resp.get("GroupDefinitions", [])
        if not any(g.get("Type") == "TAG" for g in group_defs):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "GroupDefinitions does not include a TAG dimension. The "
                    "GroupBy argument must be [{'Type': 'TAG', 'Key': '<lab tag>'}]."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

Click Apply with the defaults. The form already targets yesterday to today, GroupBy=TAG, Granularity=DAILY. Cost Explorer's 24-hour data lag means the Groups list may be empty; that is fine for the checkpoint, which validates the call shape and response structure.

</details>

<details><summary>Hint 2 (direction)</summary>

`ce.get_cost_and_usage` requires `TimePeriod` (Start + End in YYYY-MM-DD strings), `Granularity` (DAILY/MONTHLY/HOURLY), `Metrics` (a list of strings), and `GroupBy` (a list of dimension dicts). For tag-based grouping, `GroupBy=[{'Type': 'TAG', 'Key': 'labrun:lab-slug'}]` is the shape. The checkpoint inspects the response's `GroupDefinitions` to confirm TAG grouping was requested.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
resp = ce.get_cost_and_usage(
    TimePeriod={"Start": "2026-05-12", "End": "2026-05-13"},
    Granularity="DAILY",
    Metrics=["UnblendedCost"],
    GroupBy=[{"Type": "TAG", "Key": "labrun:lab-slug"}],
)
```

</details>

**Wallet check.** That one Cost Explorer call cost exactly $0.01. The lab's running total is now about 2 cents.

## Cleanup

The Budget, the Cost Anomaly monitor, and the Cost Anomaly subscription have no labrun-checks adapter in 0.6.0, so the cell below deletes them manually before `run_cleanup` removes the rest. Anomaly subscription deletes before its monitor (`delete-anomaly-monitor` fails with `MonitorInUseException` if a subscription still references it). The SNS subscription is auto-deleted with the topic; no separate manifest entry. Re-running this cell is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Manual deletes first (Budget, anomaly monitor + subscription),
# then run_cleanup runs the manifest (S3 object, S3 bucket, EC2 instance,
# SG, instance profile, role, SNS topic).

import sys

_delete_manually_managed_resources()
print("Manually-managed resources deleted (Budget, anomaly subscription, anomaly monitor).")

_rebuild_manifest()
result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if r.type == "ec2_instance")
critical_destroyed = sum(
    1 for r in CLEANUP_MANIFEST
    if r.type == "ec2_instance" and r.id not in failed_ids
)
standard_total = len(CLEANUP_MANIFEST) - critical_total
standard_destroyed = standard_total - sum(
    1 for rid in failed_ids for r in CLEANUP_MANIFEST
    if r.id == rid and r.type != "ec2_instance"
)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.02.** One Cost Explorer call at $0.01 plus a t4g.nano hour-fraction plus fractions of pennies on S3, SNS, and IAM. The two real things this lab produced are the budget guardrails and the muscle memory: the next time someone in the company runs up a four-figure surprise bill, you have the pattern in your account history that catches it before payroll calls.

## Reflection

These are not graded. They are for you.

1. Walk through what AWS does in the 24 hours after this lab's EC2 instance is launched. List the data pipeline: per-resource usage records, hourly aggregation, daily Cost and Usage Report publication, Cost Explorer ingestion, tag-based grouping availability. Where does the 24-hour lag fit in that flow, and why is it impossible to assert a non-zero cost on the same day in this lab?

2. Your team is choosing between three cost-control surfaces: (a) AWS Budgets with alarm actions, (b) Cost Anomaly Detection, (c) custom CloudWatch alarms on Billing metrics. Walk through what each one does that the others do not, and when each is the right call. Which two would you pair, and which one is redundant in most setups?

## Exam-Style Practice

**Question 1 (Domain 4, cost-control mechanism selection):**

A team's AWS bill jumped 4x last month with no architectural changes. The team wants three guardrails in place before the next billing cycle:

1. A monthly cost ceiling that fires an alert if spend crosses 80% of the ceiling.
2. Automatic detection of unusual spend patterns without manually setting thresholds.
3. Per-workload cost attribution so finance can identify which team or service drove a spike.

Which combination of AWS services satisfies all three?

A. AWS Budgets for #1 and #2, Cost and Usage Report for #3.

B. AWS Budgets for #1, Cost Anomaly Detection for #2, tag-based Cost Explorer queries for #3.

C. CloudWatch Billing alarms for all three, with custom thresholds per workload.

D. AWS Trusted Advisor for #1 and #2, Cost Explorer for #3.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong on #2: AWS Budgets handles fixed thresholds, not anomaly detection. Cost Anomaly Detection is a separate AWS service that uses ML to baseline normal spend patterns and surface deviations.
- B is correct: AWS Budgets handles the fixed-threshold alarm pattern; Cost Anomaly Detection handles the anomaly-detection-without-thresholds pattern; tag-based Cost Explorer queries handle the per-workload attribution pattern. These three services are the AWS-recommended cost-control trio.
- C is wrong: CloudWatch Billing alarms only work in us-east-1 and only support fixed-threshold alarms on the total Estimated Charges metric. They cannot do anomaly detection or per-workload attribution.
- D is wrong: AWS Trusted Advisor surfaces best-practice violations and cost-optimization recommendations (Reserved Instance suggestions, idle resource detection), not threshold alarms or anomaly detection.

</details>

**Question 2 (Domain 4, tag-based cost attribution mechanics):**

A team applies the tag `cost-center=marketing` to every resource the marketing team provisions. After 24 hours, the team runs `ce.get_cost_and_usage` with `GroupBy=[{Type: TAG, Key: cost-center}]`. The response returns zero cost for the `marketing` tag value, even though the team has confirmed the tag is applied to a running EC2 instance and an S3 bucket via the Resource Groups Tagging API. What is the most likely cause?

A. The `cost-center` tag has not been activated as a Cost Allocation Tag in the Billing console, so Cost Explorer does not surface it.

B. AWS Cost Explorer does not support tag-based grouping on user-defined tags.

C. The 24-hour lag means the data is still pending; the team needs to wait another 24 hours.

D. Cost Explorer requires the tag to be applied via CloudFormation; manually-tagged resources are excluded from cost attribution.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: user-defined tags do NOT automatically surface in Cost Explorer. The Billing console has a separate 'Cost Allocation Tags' page where tag keys must be explicitly activated. Until activated, the tag's cost data is collected but not exposed in Cost Explorer or the Cost and Usage Report. This is the single most-missed gotcha in cost-attribution setups.
- B is wrong: Cost Explorer fully supports user-defined tag grouping after the tag is activated.
- C is wrong on the symptom: 24-hour lag means the data is delayed by 24 hours, not that the tag is silently excluded. If the tag were activated and the resource tagged, the data would surface tomorrow. The fact that it returns zero specifically because the tag is inactive is the cause here.
- D is wrong: CloudFormation has nothing to do with cost-attribution eligibility. Manually-applied tags work fine if the tag key is activated as a Cost Allocation Tag.

</details>